In [1]:
import pandas as pd
import dhlab as dh

In [3]:
barn = pd.read_csv("barn.csv", low_memory=False, index_col=0)

In [10]:
barn

,dhlabid,urn,lenker,title,authors,oaiid,city,year,publisher,langs,...,ddc,genres,literaryform,kristin,sesamid,isbn10,timestamp,doctype,ocr_creator,ocr_timestamp
0,100028906,URN:NBN:no-nb_digibok_2008041604007,Vis element,I barneflokken,"Hovden , Anders / Nilssen , Jens R.",NaN,Oslo,1945.0,Fonna,mul / nno / nob,...,NaN,short story,Skjønnlitteratur,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,100439483,URN:NBN:no-nb_digibok_2021032348614,Vis element,Det går av seg sjølv,"Galåen , Johannes",NaN,Oslo,1945.0,Norli,mul / nno / nob,...,NaN,NaN,Skjønn,Skjønn,NaN,NaN,NaN,NaN,NaN,NaN
2,100573157,URN:NBN:no-nb_digibok_2007011101071,Vis element,Hittebarnet,"Baker , Samuel W. / Hagerup , Inger",NaN,Oslo,1945.0,Norsk barneblads forl.,mul / nno / nob / eng,...,NaN,NaN,Skjønn,Skjønn,NaN,NaN,NaN,NaN,NaN,NaN
3,100572304,URN:NBN:no-nb_digibok_2007010901034,Vis element,"Stor kar, son hans far : ei forteljing for gut...","Hauge , Alfred / Wangensten-Berge , Lars",NaN,Oslo,1945.0,Ansgar,nno,...,NaN,NaN,Skjønn,Skjønn,NaN,NaN,NaN,NaN,NaN,NaN
4,100572635,URN:NBN:no-nb_digibok_2007011100068,Vis element,Noreg,"Hoprekstad , Olav",NaN,NaN,1945.0,Cappelen,nno,...,NaN,NaN,Fag,Fag,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45962,100694253,URN:NBN:no-nb_pliktmonografi_000033452,NaN,De fremmede,"Houm, Nicolai / Guaragna, Luis Antonio",oai:nb.bibsys.no:999920519197002202,Oslo,2024.0,Gyldendal,nob,...,741.59481,fiction / tekst / stillbilde,Skjønnlitteratur,NaN,e74507be030d1013d85b6cfd5bd709af,NaN,20240101.0,digibok,nb,20250702.0
45963,100691466,URN:NBN:no-nb_pliktmonografi_000030369,NaN,Med blikk for humor : studiar i barnelitteratur,"Mathisen, Ingrid Nestås / Teigland, Anne-Stefi",oai:nb.bibsys.no:999920479493502202,Oslo,2024.0,Cappelen Damm akademisk,mul / nno / nob,...,809.89282,tekst,Faglitteratur,NaN,7c9d4357d1011a14604ee0602d0a5a7c,NaN,20240101.0,digibok,nb,20250702.0
45964,100692159,URN:NBN:no-nb_pliktmonografi_000031302,NaN,Nagel. 2 : Ingen vei tilbake,"Lilleeng, Sigbjørn / Ide, Christopher",oai:nb.bibsys.no:999920492497502202,Oslo,2024.0,Strand forlag,nob,...,741.59481,fiction / tekst / stillbilde,Skjønnlitteratur,NaN,20f3dc962c5fef833cb1ec5f40d30f10,NaN,20240101.0,digibok,nb,20250702.0
45965,100692060,URN:NBN:no-nb_pliktmonografi_000031199,NaN,VM i løgn,"Sjødal, Nina Anderson",oai:nb.bibsys.no:999920492997702202,Oslo,2024.0,Gyldendal,nob,...,839.8238,fiction / tekst,Skjønnlitteratur,NaN,5e86ee12c0d27eaa5e7b05c928869d0e,NaN,20240101.0,digibok,nb,20250702.0


In [6]:
from dhlab.api.dhlab_api import urn_collocation


def colloc(corp, words=['working'], before=5, after = 5, reference=None, alpha = False, samplesize= 5000):
    
    coll = pd.concat(
            [
                urn_collocation(
                    urns=list(corp.urn.values),
                    word=w,
                    before=before,
                    after=after,
                    samplesize=samplesize,
                )
                for w in words
            ]
        )[["counts"]]

    if alpha == True:
        coll = coll.loc[[x for x in coll.index if x.isalpha()]]
        if reference is not None:
            reference = reference.loc[[x for x in reference.index if x.isalpha()]]

    collocation = coll.groupby(coll.index).sum()
    if reference is not None:
        if type(reference) == pd.core.frame.DataFrame:
            """assume it has columns freq"""
            reference = reference.freq
        teller = collocation['counts'] / collocation['counts'].sum()
        divided = reference / reference.sum()
        collocation["relevance"] = (teller / divided).drop_duplicates()
    return collocation

In [21]:
barn_clean = barn[barn["urn"].notna()].copy()

r = colloc(barn_clean, words=["Hør"], before=0, after=1, samplesize=60000)

/home/larsj/miniconda3/lib/python3.11/site-packages/dhlab/api/dhlab_api.py:715: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  return pd.read_json(StringIO(r.json()))
/home/larsj/miniconda3/lib/python3.11/site-packages/dhlab/api/dhlab_api.py:715: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  return pd.read_json(StringIO(r.json()))
/home/larsj/miniconda3/lib/python3.11/site-packages/dhlab/api/dhlab_api.py:715: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing stri

In [22]:
r.loc["her"]

counts    22636
Name: her, dtype: int64